In [ ]:
load_ext jupyter_black

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
    "dpo_model",
]
big_df = pd.DataFrame([])
for model in models:
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df = df.loc[
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
    ]
    df["sycophancy"] = (
        df["model_answer_constrained_no_flipped_story"]
        > df["model_answer_constrained_yes_flipped_story"]
    )
    df["model"] = model
    big_df = pd.concat([df, big_df], ignore_index=True)

In [ ]:
# baseline
plt.hlines(
    big_df[pd.isna(big_df["persona_type"])]
    .groupby("model")["sycophancy"]
    .mean()
    .tolist(),
    xmin=-0.5,
    xmax=10,
    colors=["red", "orange", "blue", "green"],
    linestyles="dashed",
)
ax = sns.barplot(big_df, x="persona_type", y="sycophancy", hue="model")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("Type of information shared")

In [ ]:
big_df[pd.isna(big_df["persona_type"])].groupby("model")["sycophancy"].mean()

In [ ]:
# min
plt.hlines(
    big_df.loc[big_df["model"] != "dpo_model"]
    .groupby(["model", "persona_id", "persona_type"])["sycophancy"]
    .mean()
    .reset_index()
    .sort_values(by="model")
    .groupby(["model"])["sycophancy"]
    .min()
    .tolist(),
    xmin=-0.5,
    xmax=10,
    colors=["blue", "orange", "green", "red", "purple"],
    linestyles="dotted",
)
# baseline
plt.hlines(
    big_df.loc[big_df["model"] != "dpo_model"][pd.isna(big_df["persona_type"])]
    .sort_values(by="model")
    .groupby("model")["sycophancy"]
    .mean()
    .tolist(),
    xmin=-0.5,
    xmax=10,
    colors=["blue", "orange", "green", "red", "purple"],
    linestyles="dashed",
)
# max
plt.hlines(
    big_df.loc[big_df["model"] != "dpo_model"]
    .groupby(["model", "persona_id", "persona_type"])["sycophancy"]
    .mean()
    .reset_index()
    .sort_values(by="model")
    .groupby(["model"])["sycophancy"]
    .max()
    .tolist(),
    xmin=-0.5,
    xmax=10,
    colors=["blue", "orange", "green", "red", "purple"],
    linestyles="dotted",
)
ax = sns.barplot(
    big_df.loc[big_df["model"] != "dpo_model"].sort_values(by="model"),
    x="persona_type",
    y="sycophancy",
    hue="model",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("Type of information shared")

In [ ]:
ax = sns.barplot(big_df, x="persona_type", y="sycophancy", hue="model")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("Type of information shared")
df = (
    big_df.groupby(["model", "persona_type"])["sycophancy"]
    .mean()
    .sort_values()
    .reset_index()
)
for m in range(len(models)):
    label_dict = {
        df[df["model"] == models[m]]["persona_type"].iloc[i]: int(
            df[df["model"] == models[m]].rank(numeric_only=True).iloc[i].item()
        )
        for i in range(10)
    }
    ax.bar_label(
        ax.containers[m],
        labels=[label_dict[x.get_text()] for x in ax.get_xticklabels()],
        padding=3,
    )

In [ ]:
big_df.groupby(["model", "persona_id", "persona_type"])[
    "sycophancy"
].mean().reset_index().groupby(["model", "persona_type"])["sycophancy"].agg(
    ["min", "max"]
)